# 03 — Entrega 1: resultados

Este notebook **só carrega** resultados salvos (`outputs/` e `docs/figuras/`): não treina nada e
abre em poucos segundos. Ordem: execução → dados → EDA → pré-processamento → baselines →
métricas → erros → teste externo → custo → limitações.

In [ ]:
# ===== Célula de ambiente (igual em todos os notebooks) =====
# Local (kernel .venv): usa o que já está em data/ e outputs/; não instala nem baixa nada.
# Colab (extensão do VS Code ou navegador): monta o Drive, clona/atualiza o repositório,
# instala as dependências e prepara o que este notebook precisa. Preencha REPO_URL para o Colab.
REPO_URL = "https://github.com/SEU_USUARIO/pathoshift.git"
BRANCH = "main"

import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/pathoshift")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ambiente import preparar
preparar(ROOT, precisa_dados=False, precisa_features=False, precisa_resultados=True)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

OUT, FIG_EDA = ROOT / "outputs", ROOT / "docs" / "figuras"
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
run = json.loads((OUT / "run_info.json").read_text(encoding="utf-8"))
MODELOS = {"logreg": "Regressão logística", "cnn": "SimpleCNN"}

def fig(path, width=900):
    display(Image(str(path), width=width))

def ms(d):
    return "—" if d["media"] is None else f"{d['media']:.3f} ± {d['desvio']:.3f}"

## 1. Execução: configuração, ambiente e registro do teste externo

In [ ]:
print("seed:", run["seed"], "| GPU:", run["gpu"], "| plataforma:", run["plataforma"])
print("versões:", {k: v for k, v in run["versoes"].items() if k in ("python", "torch", "scikit-learn", "opencv-python-headless", "numpy")})
print("\nTeste externo:", json.dumps(run.get("teste_externo", "ainda não executado"), indent=2, ensure_ascii=False))

## 2. Dados: patches, tumores, pacientes e lâminas

In [ ]:
display(pd.read_csv(OUT / "tabelas" / "resumo_por_centro.csv", index_col=0))
display(pd.read_csv(OUT / "tabelas" / "resumo_por_fold.csv", index_col=0))

## 3. EDA: aparência por centro e tumor de região única

In [ ]:
fig(FIG_EDA / "01_patches_por_centro.png")
fig(FIG_EDA / "11_regiao_unica_tumor.png")

## 4. Pré-processamento: máscara de tecido (S > 20 e não Otsu) e máscara de núcleos

In [ ]:
fig(FIG_EDA / "03_histograma_S_otsu.png")
fig(FIG_EDA / "04_mascara_tecido.png", 800)
fig(FIG_EDA / "05_mascara_nucleos.png", 800)

## 5. Baselines: validação cruzada (4 folds por paciente), média ± desvio

In [ ]:
chaves = ["accuracy", "balanced_accuracy", "f1_macro", "auc", "recall_tumor", "recall_normal", "precision_tumor", "f1_tumor"]
display(pd.DataFrame({MODELOS[m]: {k: ms(metrics["modelos"][m]["cv"]["por_patch"][k]) for k in chaves}
                      for m in MODELOS}).rename(index={"recall_normal": "especificidade"}))
print("Por paciente (cada paciente pesa igual):")
display(pd.DataFrame({MODELOS[m]: {k: ms(v) for k, v in metrics["modelos"][m]["cv"]["por_paciente"].items()}
                      for m in MODELOS}))

In [ ]:
print("Por centro (predições de validação dos 4 folds juntas; centros sem tumor: só especificidade):")
cols = ["centre_id", "patches", "tumor", "recall_tumor", "especificidade", "acuracia_balanceada", "auc"]
display(pd.concat({MODELOS[m]: pd.DataFrame(metrics["modelos"][m]["cv"]["por_centro"])[cols].set_index("centre_id")
                   for m in MODELOS}, axis=1).round(3))

## 6. Métricas: comparação, matrizes de confusão e curvas da CNN

In [ ]:
fig(OUT / "figuras" / "comparacao_modelos.png")
for m in MODELOS:
    fig(OUT / "figuras" / f"matriz_confusao_{m}.png", 600)
fig(OUT / "figuras" / "curvas_cnn.png")

## 7. Erros: recall por paciente com tumor e exemplos de FP/FN

In [ ]:
det = {MODELOS[m]: pd.DataFrame(metrics["modelos"][m]["cv"]["detalhe_pacientes"]).set_index("patient_id")
       for m in MODELOS}
tum = pd.concat({k: v.loc[v["tumor"] > 0, ["patches", "tumor", "recall_tumor", "especificidade"]] for k, v in det.items()}, axis=1)
display(tum.round(3))
for m in MODELOS:
    fig(OUT / "figuras" / f"erros_{m}.png")

## 8. Teste externo — centro 2 (executado uma única vez, com a configuração congelada)

In [ ]:
if all("teste_externo" in metrics["modelos"][m] for m in MODELOS):
    te = {MODELOS[m]: metrics["modelos"][m]["teste_externo"] for m in MODELOS}
    display(pd.DataFrame({k: {**{c: v["por_patch"][c] for c in chaves}, **v["por_paciente"]} for k, v in te.items()})
            .rename(index={"recall_normal": "especificidade"}).round(3))
    display(pd.concat({k: pd.DataFrame(v["detalhe_pacientes"]).set_index("patient_id")[["patches", "tumor", "recall_tumor", "especificidade"]]
                       for k, v in te.items()}, axis=1).round(3))
else:
    print("Teste externo ainda não executado.")

## 9. Custo computacional

In [ ]:
etapas = pd.DataFrame(run["etapas"]).T[["segundos", "quando"]]
display(etapas)
print(f"Total: {etapas['segundos'].astype(float).sum() / 60:.1f} min | SimpleCNN: 3.267.778 parâmetros")

## 10. Limitações
- Poucos pacientes com tumor no desenvolvimento (8, sendo só 5 com 300 patches): o desvio entre folds é grande.
- Pacientes 16, 17 (dev) e 41 (centro 2) têm os 300 patches de tumor numa região de ~300×200 px, quase a mesma imagem.
- Mudança de domínio já no dev: a CNN marca como tumor a maioria dos normais dos centros 4 e 5.
- Máscaras de núcleos falham em tecido escuro/denso; limiares escolhidos à mão; limiar de decisão fixo em 0,5.
- A amostra exploratória que embasou os limiares de S e Sobel incluía patches (sem rótulos) do centro 2.